In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/processed")

df = pd.read_csv(DATA_PATH / "analysis_ready.csv")

df["date"] = pd.to_datetime(df["date"])

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (365200, 21)


In [5]:
weekly_sales = (
    df.groupby([
        "sku",
        pd.Grouper(key="date", freq="W-SUN")
    ])["units_sold"]
    .sum()
    .reset_index()
)

weekly_sales = weekly_sales.rename(
    columns={"units_sold": "actual_demand"}
)

weekly_sales = weekly_sales.sort_values(
    ["sku", "date"]
).reset_index(drop=True)

print("Weekly dataset shape:", weekly_sales.shape)

weekly_sales.head()

Weekly dataset shape: (52400, 3)


,sku,date,actual_demand
0,SKU0001,2021-01-03,0
1,SKU0001,2021-01-10,0
2,SKU0001,2021-01-17,0
3,SKU0001,2021-01-24,0
4,SKU0001,2021-01-31,0


In [6]:
all_weeks = pd.date_range(
    weekly_sales["date"].min(),
    weekly_sales["date"].max(),
    freq="W-SUN"
)

all_skus = weekly_sales["sku"].unique()

complete_index = pd.MultiIndex.from_product(
    [all_skus, all_weeks],
    names=["sku", "date"]
)

weekly_sales = (
    weekly_sales
    .set_index(["sku", "date"])
    .reindex(complete_index)
    .reset_index()
)

weekly_sales["actual_demand"] = (
    weekly_sales["actual_demand"].fillna(0)
)

weekly_sales = weekly_sales.sort_values(
    ["sku", "date"]
).reset_index(drop=True)

print("Complete weekly dataset:", weekly_sales.shape)

Complete weekly dataset: (52400, 3)


In [7]:
def wape(actual, forecast):
    denominator = np.sum(np.abs(actual))
    
    if denominator == 0:
        return np.nan
    
    return np.sum(np.abs(actual - forecast)) / denominator

In [8]:
weekly_sales["baseline_forecast"] = (
    weekly_sales
    .groupby("sku")["actual_demand"]
    .shift(52)
)

In [9]:
cutoff_date = (
    weekly_sales["date"].max()
    - pd.Timedelta(weeks=26)
)

train = weekly_sales[
    weekly_sales["date"] <= cutoff_date
].copy()

test = weekly_sales[
    weekly_sales["date"] > cutoff_date
].copy()

print("Training period:")
print(train["date"].min(), "to", train["date"].max())

print("\nTesting period:")
print(test["date"].min(), "to", test["date"].max())

print("\nTrain rows:", len(train))
print("Test rows:", len(test))

Training period:
2021-01-03 00:00:00 to 2025-07-06 00:00:00

Testing period:
2025-07-13 00:00:00 to 2026-01-04 00:00:00

Train rows: 47200
Test rows: 5200


In [10]:
baseline_test = test.dropna(
    subset=["baseline_forecast"]
).copy()

baseline_wape = wape(
    baseline_test["actual_demand"],
    baseline_test["baseline_forecast"]
)

print(
    f"Seasonal-Naive Baseline WAPE: "
    f"{baseline_wape:.4f}"
)

print(
    f"Seasonal-Naive Baseline WAPE: "
    f"{baseline_wape * 100:.2f}%"
)

Seasonal-Naive Baseline WAPE: 0.1165
Seasonal-Naive Baseline WAPE: 11.65%


In [11]:
weekly_sales = weekly_sales.sort_values(
    ["sku", "date"]
).reset_index(drop=True)

weekly_sales["lag_1"] = (
    weekly_sales.groupby("sku")["actual_demand"].shift(1)
)

weekly_sales["lag_2"] = (
    weekly_sales.groupby("sku")["actual_demand"].shift(2)
)

weekly_sales["lag_4"] = (
    weekly_sales.groupby("sku")["actual_demand"].shift(4)
)

weekly_sales["lag_52"] = (
    weekly_sales.groupby("sku")["actual_demand"].shift(52)
)

In [12]:
weekly_sales["rolling_4_week"] = (
    weekly_sales
    .groupby("sku")["actual_demand"]
    .transform(
        lambda x: x.shift(1).rolling(4).mean()
    )
)

weekly_sales["rolling_12_week"] = (
    weekly_sales
    .groupby("sku")["actual_demand"]
    .transform(
        lambda x: x.shift(1).rolling(12).mean()
    )
)

print("Feature engineering completed.")

Feature engineering completed.


In [13]:
weekly_sales["year"] = weekly_sales["date"].dt.year
weekly_sales["month"] = weekly_sales["date"].dt.month
weekly_sales["quarter"] = weekly_sales["date"].dt.quarter

In [14]:
calendar_weekly = (
    df[["date", "season", "holiday_flag", "promotion_event"]]
    .drop_duplicates()
    .set_index("date")
    .resample("W-SUN")
    .agg({
        "season": lambda x: x.mode()[0] if not x.mode().empty else "Unknown",
        "holiday_flag": "max",
        "promotion_event": lambda x: 1 if (x != "None").any() else 0
    })
    .reset_index()
)

In [15]:
weekly_sales = weekly_sales.merge(
    calendar_weekly,
    on="date",
    how="left"
)

print("Calendar features added.")

Calendar features added.


In [16]:
print(
    weekly_sales[
        [
            "date",
            "year",
            "month",
            "quarter",
            "season",
            "holiday_flag",
            "promotion_event"
        ]
    ].head(10)
)

        date  year  month  quarter  season  holiday_flag  promotion_event
0 2021-01-03  2021      1        1  Winter             0                1
1 2021-01-10  2021      1        1  Winter             0                1
2 2021-01-17  2021      1        1  Winter             0                1
3 2021-01-24  2021      1        1  Winter             0                1
4 2021-01-31  2021      1        1  Winter             0                1
5 2021-02-07  2021      2        1  Winter             0                1
6 2021-02-14  2021      2        1  Winter             0                1
7 2021-02-21  2021      2        1  Winter             0                1
8 2021-02-28  2021      2        1  Winter             0                1
9 2021-03-07  2021      3        1  Spring             0                1


In [17]:
print(weekly_sales.shape)
print(weekly_sales.columns.tolist())

(52400, 16)
['sku', 'date', 'actual_demand', 'baseline_forecast', 'lag_1', 'lag_2', 'lag_4', 'lag_52', 'rolling_4_week', 'rolling_12_week', 'year', 'month', 'quarter', 'season', 'holiday_flag', 'promotion_event']


In [18]:
print(weekly_sales.isnull().sum())


sku                      0
date                     0
actual_demand            0
baseline_forecast    10400
lag_1                  200
lag_2                  400
lag_4                  800
lag_52               10400
rolling_4_week         800
rolling_12_week       2400
year                     0
month                    0
quarter                  0
season                   0
holiday_flag             0
promotion_event          0
dtype: int64


In [19]:
model_df = weekly_sales.dropna(
    subset=[
        "lag_1",
        "lag_2",
        "lag_4",
        "lag_52",
        "rolling_4_week",
        "rolling_12_week"
    ]
).copy()

In [20]:
season_mapping = {
    "Spring": 0,
    "Summer": 1,
    "Monsoon": 2,
    "Winter": 3
}

model_df["season_encoded"] = model_df["season"].map(season_mapping)

print(model_df[["season", "season_encoded"]].drop_duplicates())

     season  season_encoded
52   Winter               3
61   Spring               0
74   Summer               1
87  Monsoon               2


In [21]:
features = [
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_52",
    "rolling_4_week",
    "rolling_12_week",
    "year",
    "month",
    "quarter",
    "season_encoded",
    "holiday_flag",
    "promotion_event"
]

target = "actual_demand"

In [22]:
print("Features:")
print(features)

print("\nModel data shape:")
print(model_df.shape)

print("\nMissing values in model features:")
print(model_df[features].isnull().sum())

Features:
['lag_1', 'lag_2', 'lag_4', 'lag_52', 'rolling_4_week', 'rolling_12_week', 'year', 'month', 'quarter', 'season_encoded', 'holiday_flag', 'promotion_event']

Model data shape:
(42000, 17)

Missing values in model features:
lag_1              0
lag_2              0
lag_4              0
lag_52             0
rolling_4_week     0
rolling_12_week    0
year               0
month              0
quarter            0
season_encoded     0
holiday_flag       0
promotion_event    0
dtype: int64


In [23]:
model_df = model_df.sort_values(
    ["date", "sku"]
).reset_index(drop=True)

In [24]:
cutoff_date = (
    model_df["date"].max()
    - pd.Timedelta(weeks=26)
)

train_df = model_df[
    model_df["date"] <= cutoff_date
].copy()

test_df = model_df[
    model_df["date"] > cutoff_date
].copy()

In [25]:
X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

In [26]:
print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)

print("\nTraining period:")
print(train_df["date"].min(), "to", train_df["date"].max())

print("\nTesting period:")
print(test_df["date"].min(), "to", test_df["date"].max())

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Training data: (36800, 17)
Testing data: (5200, 17)

Training period:
2022-01-02 00:00:00 to 2025-07-06 00:00:00

Testing period:
2025-07-13 00:00:00 to 2026-01-04 00:00:00

X_train shape: (36800, 12)
X_test shape: (5200, 12)


In [27]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created.")

Random Forest model created.


In [28]:
rf_model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [29]:
y_pred = rf_model.predict(X_test)

print("Predictions created.")

Predictions created.


In [30]:
y_pred = np.maximum(y_pred, 0)

In [31]:
model_wape = wape(y_test.values, y_pred)

print(f"Random Forest WAPE: {model_wape:.4f}")
print(f"Random Forest WAPE: {model_wape * 100:.2f}%")

Random Forest WAPE: 0.1030
Random Forest WAPE: 10.30%


In [32]:
print(f"Seasonal-Naive Baseline WAPE: {baseline_wape * 100:.2f}%")
print(f"Random Forest Model WAPE: {model_wape * 100:.2f}%")

Seasonal-Naive Baseline WAPE: 11.65%
Random Forest Model WAPE: 10.30%


In [55]:
improvement = (
    (baseline_wape - model_wape)
    / baseline_wape
) * 100

print(f"Model Improvement: {improvement:.2f}%")

Model Improvement: 11.59%


Stockout & Overstock Risk Scoring.

In [33]:
risk_df = test_df[[
    "sku",
    "date",
    "actual_demand"
]].copy()

risk_df["forecast_demand"] = y_pred

print(risk_df.head())

           sku       date  actual_demand  forecast_demand
36800  SKU0001 2025-07-13            203       225.763639
36801  SKU0002 2025-07-13            436       434.960409
36802  SKU0003 2025-07-13            179       126.246522
36803  SKU0004 2025-07-13             96        87.126435
36804  SKU0005 2025-07-13             27        41.493381


In [34]:
sku_forecast = (
    risk_df
    .groupby("sku", as_index=False)
    .agg(
        forecast_demand=("forecast_demand", "sum"),
        actual_demand=("actual_demand", "sum")
    )
)

print(sku_forecast.head())

       sku  forecast_demand  actual_demand
0  SKU0001      5653.025742           5376
1  SKU0002     11560.513733          11392
2  SKU0003      3315.489354           3285
3  SKU0004      2251.584186           2150
4  SKU0005       984.910808            954


In [35]:
print(df.columns.tolist())

['date', 'sku', 'units_sold', 'revenue', 'unit_price', 'promotion_flag', 'year', 'month', 'week', 'day_of_week', 'quarter', 'is_weekend', 'season', 'holiday_flag', 'promotion_event', 'category', 'subcategory', 'launch_date', 'unit_cost', 'list_price', 'lead_time_days']


In [40]:
import pandas as pd
DATA_PATH = Path("../data/raw")

inventory_snapshots = pd.read_csv(DATA_PATH / "inventory_snapshots.csv")

print(inventory_snapshots.head())
print(inventory_snapshots.columns.tolist())

         date      sku  on_hand_units  on_order_units  lead_time_days  \
0  2021-01-03  SKU0001              0               1              26   
1  2021-01-03  SKU0002              0               3              24   
2  2021-01-03  SKU0003             10               5              19   
3  2021-01-03  SKU0004              2              11              28   
4  2021-01-03  SKU0005              7               0               8   

   reorder_point_units  inventory_value  
0                   10             0.00  
1                   10             0.00  
2                   10          8710.20  
3                   10          5821.22  
4                   10         16508.66  
['date', 'sku', 'on_hand_units', 'on_order_units', 'lead_time_days', 'reorder_point_units', 'inventory_value']


In [41]:
inventory_snapshots["date"] = pd.to_datetime(
    inventory_snapshots["date"]
)

latest_inventory_date = inventory_snapshots["date"].max()

latest_inventory = inventory_snapshots[
    inventory_snapshots["date"] == latest_inventory_date
].copy()

print("Latest inventory date:", latest_inventory_date)
print(latest_inventory.head())

Latest inventory date: 2025-12-28 00:00:00
            date      sku  on_hand_units  on_order_units  lead_time_days  \
52000 2025-12-28  SKU0001            500             450              26   
52001 2025-12-28  SKU0002           1137            1023              24   
52002 2025-12-28  SKU0003            553             209              19   
52003 2025-12-28  SKU0004            369             132              28   
52004 2025-12-28  SKU0005            708             189               8   

       reorder_point_units  inventory_value  
52000                 1053       1367440.00  
52001                 2201       2420843.55  
52002                  476        481674.06  
52003                  468       1074015.09  
52004                   68       1669733.04  


In [42]:
risk_summary = sku_forecast.merge(
    latest_inventory[["sku", "on_hand_units"]],
    on="sku",
    how="left"
)

print(risk_summary.head())

       sku  forecast_demand  actual_demand  on_hand_units
0  SKU0001      5653.025742           5376            500
1  SKU0002     11560.513733          11392           1137
2  SKU0003      3315.489354           3285            553
3  SKU0004      2251.584186           2150            369
4  SKU0005       984.910808            954            708


In [43]:
risk_summary["stock_gap_units"] = (
    risk_summary["forecast_demand"]
    - risk_summary["on_hand_units"]
)

risk_summary["stockout_risk"] = (
    risk_summary["stock_gap_units"] > 0
)

In [44]:
risk_summary["overstock_risk"] = (
    risk_summary["on_hand_units"]
    > risk_summary["forecast_demand"] * 1.5
)

In [45]:
def recommend_action(row):

    if row["stockout_risk"]:
        return "REORDER"

    elif row["overstock_risk"]:
        return "MARKDOWN / CLEAR"

    else:
        return "LEAVE ALONE"

In [46]:
risk_summary["recommended_action"] = (
    risk_summary.apply(
        recommend_action,
        axis=1
    )
)

print(
    risk_summary[
        [
            "sku",
            "forecast_demand",
            "on_hand_units",
            "stockout_risk",
            "overstock_risk",
            "recommended_action"
        ]
    ].head(10)
)

       sku  forecast_demand  on_hand_units  stockout_risk  overstock_risk  \
0  SKU0001      5653.025742            500           True           False   
1  SKU0002     11560.513733           1137           True           False   
2  SKU0003      3315.489354            553           True           False   
3  SKU0004      2251.584186            369           True           False   
4  SKU0005       984.910808            708           True           False   
5  SKU0006      4577.660652            761           True           False   
6  SKU0007      3898.535704            655           True           False   
7  SKU0008      4273.486676            748           True           False   
8  SKU0009      3653.022458            627           True           False   
9  SKU0010      5424.693299            487           True           False   

  recommended_action  
0            REORDER  
1            REORDER  
2            REORDER  
3            REORDER  
4            REORDER  
5            R

In [48]:
# Create priority score
risk_summary["priority_score"] = 0.0

# Create stockout mask
stockout_mask = risk_summary["stockout_risk"] == True

# Stockout priority = stock gap
risk_summary.loc[
    stockout_mask,
    "priority_score"
] = risk_summary.loc[
    stockout_mask,
    "stock_gap_units"
]

# Create overstock mask
overstock_mask = risk_summary["overstock_risk"] == True

# Overstock priority = excess inventory
risk_summary.loc[
    overstock_mask,
    "priority_score"
] = (
    risk_summary.loc[
        overstock_mask,
        "on_hand_units"
    ]
    -
    risk_summary.loc[
        overstock_mask,
        "forecast_demand"
    ]
)

# Sort by priority
risk_summary = risk_summary.sort_values(
    "priority_score",
    ascending=False
)

print(
    risk_summary[
        [
            "sku",
            "forecast_demand",
            "on_hand_units",
            "stock_gap_units",
            "stockout_risk",
            "overstock_risk",
            "priority_score"
        ]
    ].head(10)
)

         sku  forecast_demand  on_hand_units  stock_gap_units  stockout_risk  \
181  SKU0182     14047.470723           1297     12750.470723           True   
98   SKU0099     13241.173999           1206     12035.173999           True   
38   SKU0039     12971.686836           1226     11745.686836           True   
180  SKU0181     12819.873731           1240     11579.873731           True   
30   SKU0031     12296.731478           1132     11164.731478           True   
168  SKU0169     11804.456275           1092     10712.456275           True   
1    SKU0002     11560.513733           1137     10423.513733           True   
150  SKU0151     11510.138238           1090     10420.138238           True   
17   SKU0018     10757.575946           1019      9738.575946           True   
167  SKU0168     10594.249008            958      9636.249008           True   

     overstock_risk  priority_score  
181           False    12750.470723  
98            False    12035.173999  
38   

In [50]:
import joblib

joblib.dump(rf_model, "../models/random_forest_model.pkl")

print("Model saved successfully.")

Model saved successfully.


In [52]:
risk_summary.to_csv(
    "../outputs/risk_summary.csv",
    index=False
)

print("Risk summary saved.")

Risk summary saved.


In [53]:
forecast_output = test_df[
    ["sku", "date", "actual_demand"]
].copy()

forecast_output["forecast_demand"] = y_pred

forecast_output.to_csv(
    "../outputs/forecast_results.csv",
    index=False
)

print("Forecast results saved.")

Forecast results saved.


In [54]:
performance = pd.DataFrame({
    "Model": [
        "Seasonal Naive",
        "Random Forest"
    ],
    "WAPE": [
        baseline_wape,
        model_wape
    ]
})

performance.to_csv(
    "../outputs/model_performance.csv",
    index=False
)

performance

,Model,WAPE
0,Seasonal Naive,0.116512
1,Random Forest,0.103013
